# LeetCode 1983 — Widest Pair of Indices With Equal Range Sum bằng Python

Link đề: https://leetcode.com/problems/widest-pair-of-indices-with-equal-range-sum/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def widestPairOfIndices(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Notebook đi từ **thử mọi cặp `(i, j)` và cộng lại từ đầu** `O(n³)` → **tổng tiền tố** `O(n²)` → **hiệu tiền tố + bảng băm lần xuất hiện đầu tiên** `O(n)` → **mảng thay cho bảng băm** (hiệu nằm trong `[−n, n]`).

## 1. Mục tiêu học

1. Biến điều kiện "tổng đoạn của hai mảng bằng nhau" thành "tổng đoạn của mảng **hiệu** `d = nums1 − nums2` bằng 0".
2. Dùng **tổng tiền tố**: `sum(d[i..j]) = P[j+1] − P[i]` ⇒ đoạn hợp lệ ⇔ `P[i] = P[j+1]`.
3. Tìm đoạn **dài nhất** có tổng 0: với mỗi giá trị tiền tố, lưu **chỉ số xuất hiện đầu tiên** (kèm `P[0] = 0` ở chỉ số `−1`).
4. Thay `dict` bằng mảng có độ lệch (`offset`) khi miền giá trị nhỏ.
5. Kiểm thử bằng vét cạn và nhận diện các lỗi lệch một.

## 2. Đề bài & ví dụ

Cho hai mảng nhị phân `nums1`, `nums2` (đánh số từ 0) cùng độ dài `n`. Tìm cặp chỉ số **rộng nhất** `(i, j)` với `i ≤ j` sao cho

`nums1[i] + … + nums1[j] == nums2[i] + … + nums2[j]`.

Độ rộng là `j − i + 1`. Trả về độ rộng lớn nhất; nếu không có cặp nào, trả về `0`.

**Ví dụ 1:** `nums1 = [1,1,0,1]`, `nums2 = [0,1,1,0]` → `3` (`i = 1`, `j = 3`: `1 + 0 + 1 = 1 + 1 + 0 = 2`).

**Ví dụ 2:** `nums1 = [0,1]`, `nums2 = [1,1]` → `1` (`i = j = 1`).

**Ví dụ 3:** `nums1 = [0]`, `nums2 = [1]` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`, các phần tử là `0` hoặc `1`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [([1, 1, 0, 1], [0, 1, 1, 0], 3), ([0, 1], [1, 1], 1), ([0], [1], 0)]

for a, b, exp in examples:
    d = [x - y for x, y in zip(a, b)]
    print(f"nums1={a}, nums2={b}: d = nums1 - nums2 = {d} -> kỳ vọng {exp}")

nums1=[1, 1, 0, 1], nums2=[0, 1, 1, 0]: d = nums1 - nums2 = [1, 0, -1, 1] -> kỳ vọng 3
nums1=[0, 1], nums2=[1, 1]: d = nums1 - nums2 = [-1, 0] -> kỳ vọng 1
nums1=[0], nums2=[1]: d = nums1 - nums2 = [-1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 Mảng hiệu

`sum(nums1[i..j]) − sum(nums2[i..j]) = sum(d[i..j])` với `d[k] = nums1[k] − nums2[k] ∈ {−1, 0, 1}`.

In [2]:
a, b = [1, 1, 0, 1], [0, 1, 1, 0]
d = [x - y for x, y in zip(a, b)]
print("d =", d, "| sum(d[1..3]) =", sum(d[1:4]), "= sum(a[1..3]) - sum(b[1..3]) =", sum(a[1:4]) - sum(b[1:4]))

d = [1, 0, -1, 1] | sum(d[1..3]) = 0 = sum(a[1..3]) - sum(b[1..3]) = 0


### 3.2 Tổng tiền tố

`P[0] = 0`, `P[k] = d[0] + … + d[k−1]`. Tổng đoạn `d[i..j] = P[j+1] − P[i]`.

In [3]:
P = list(accumulate(d, initial=0))
print("P =", P, "| tổng d[1..3] = P[4] - P[1] =", P[4] - P[1])

P = [0, 1, 1, 0, 1] | tổng d[1..3] = P[4] - P[1] = 0


### 3.3 `dict.setdefault` và "lần xuất hiện đầu tiên"

Muốn ghi nhớ chỉ số **đầu tiên** của mỗi giá trị: chỉ gán khi khóa **chưa có**.

In [4]:
first = {}
for i, v in enumerate([3, 1, 3, 2, 1]):
    first.setdefault(v, i)
print(first)

{3: 0, 1: 1, 2: 3}


### 3.4 Mảng với độ lệch (offset)

Giá trị tiền tố nằm trong `[−n, n]` ⇒ dùng mảng dài `2n + 1`, chỉ số `v + n`.

In [5]:
n = 4
arr = [None] * (2 * n + 1)
for v in [-2, 0, 3]:
    arr[v + n] = f"đã gặp {v}"
print(arr)

[None, None, 'đã gặp -2', None, 'đã gặp 0', None, None, 'đã gặp 3', None]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `d = [1, 0, −1, 1]`, `P = [0, 1, 1, 0, 1]` (chỉ số `0..4`).

Đoạn `d[i..j]` có tổng 0 ⇔ `P[i] == P[j+1]`. Các cặp bằng nhau trong `P`:
- giá trị `0` ở chỉ số `0` và `3` ⇒ đoạn `d[0..2]`, dài 3;
- giá trị `1` ở chỉ số `1`, `2`, `4` ⇒ xa nhất `1` và `4` ⇒ đoạn `d[1..3]`, dài 3.

Đáp án `3`. Để có đoạn **dài nhất** với giá trị `v`: lấy chỉ số **đầu tiên** và **cuối cùng** của `v` trong `P`.

In [6]:
from collections import defaultdict
pos = defaultdict(list)
for k, v in enumerate(P):
    pos[v].append(k)
for v, ks in pos.items():
    print(f"P = {v} tại {ks} -> đoạn dài nhất {ks[-1] - ks[0]}")

P = 0 tại [0, 3] -> đoạn dài nhất 3
P = 1 tại [1, 2, 4] -> đoạn dài nhất 3


Vẽ đường gấp khúc của hiệu tiền tố `P` cho một ví dụ dài hơn (tự dựng): hai điểm **cùng độ cao** trên đường ⇒ đoạn giữa chúng có tổng hiệu bằng 0. Đoạn rộng nhất ứng với hai điểm cùng độ cao **xa nhau nhất**.

In [7]:
a = [1, 1, 0, 1, 0, 0, 1, 0]
b = [0, 0, 0, 1, 1, 1, 0, 1]
P = list(accumulate((x - y for x, y in zip(a, b)), initial=0))
lo, hi = min(P), max(P)
for level in range(hi, lo - 1, -1):
    print(f"  {level:>2} | " + "".join(" ● " if v == level else " · " for v in P))
print("       " + "".join(f"{k:^3}" for k in range(len(P))))
pos = {}
for k, v in enumerate(P):
    pos.setdefault(v, []).append(k)
print("  cặp cùng độ cao xa nhất:", {v: (ks[0], ks[-1]) for v, ks in pos.items()}, "-> đáp án", max(ks[-1] - ks[0] for ks in pos.values()))

   2 |  ·  ·  ●  ●  ●  ·  ·  ·  · 
   1 |  ·  ●  ·  ·  ·  ●  ·  ●  · 
   0 |  ●  ·  ·  ·  ·  ·  ●  ·  ● 
        0  1  2  3  4  5  6  7  8 
  cặp cùng độ cao xa nhất: {0: (0, 8), 1: (1, 7), 2: (2, 4)} -> đáp án 8


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp, cộng lại từ đầu

In [8]:
def widest_brute(nums1, nums2):
    n = len(nums1)
    best = 0
    for i in range(n):
        for j in range(i, n):
            if sum(nums1[i:j + 1]) == sum(nums2[i:j + 1]):
                best = max(best, j - i + 1)
    return best

for a, b, exp in examples:
    print(widest_brute(a, b), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n³)` — mỗi trong `O(n²)` cặp tính tổng `O(n)`.

**Nút thắt:** tổng đoạn tính lại từ đầu ⇒ dùng tiền tố (`O(1)` mỗi đoạn) — rồi tiến thêm: không cần duyệt mọi cặp, chỉ cần **tra** chỉ số đầu tiên có cùng tiền tố.

## 6. Ý tưởng thuật toán / bước đột phá

Đoạn `[i, j]` hợp lệ ⇔ `P[i] = P[j + 1]`. Cố định đầu phải `r = j + 1`, đoạn dài nhất kết thúc ở `j` dùng `i` **nhỏ nhất** với `P[i] = P[r]` ⇒ chỉ cần lưu `first[v]` = chỉ số nhỏ nhất có `P = v`.

Duyệt `r = 1..n` (tương ứng `j = 0..n−1`), giữ `s = P[r]`:
- **Nếu** `s` đã có trong `first` **thì** cập nhật `ans = max(ans, r − first[s])`.
- **Ngược lại** ghi `first[s] = r`.

Khởi tạo `first[0] = 0` (tức "tiền tố rỗng"); viết theo chỉ số `j` thì `first[0] = −1` và độ dài `j − first[s]`.

**Bất biến:** khi xử lý `j`, `first` chứa chỉ số đầu tiên của mọi giá trị tiền tố `P[0..j]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng tiền tố, thử mọi cặp `O(n²)`

In [9]:
def widest_prefix_pairs(nums1, nums2):
    n = len(nums1)
    P = list(accumulate((x - y for x, y in zip(nums1, nums2)), initial=0))
    best = 0
    for i in range(n):
        for j in range(n - 1, i - 1 + best, -1):     # thử j từ xa nhất, dừng khi không thể vượt best
            if P[j + 1] == P[i]:
                best = j - i + 1
                break
    return best

for a, b, exp in examples:
    print(widest_prefix_pairs(a, b), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n²)` xấu nhất (có cắt nhánh khi đoạn không thể dài hơn `best`), `O(n)` bộ nhớ.

### 7.2 Cách 3 — Bảng băm chỉ số đầu tiên (lời giải chính)

In [10]:
def widest_hash(nums1, nums2):
    first = {0: -1}
    s = ans = 0
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first:
            ans = max(ans, j - first[s])
        else:
            first[s] = j
    return ans

for a, b, exp in examples:
    print(widest_hash(a, b), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n)` thời gian trung bình, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Mảng có độ lệch thay cho bảng băm

In [11]:
def widest_array(nums1, nums2):
    n = len(nums1)
    NONE = n + 5
    first = [NONE] * (2 * n + 1)
    first[0 + n] = -1
    s = ans = 0
    for j in range(n):
        s += nums1[j] - nums2[j]
        k = s + n
        if first[k] != NONE:
            if j - first[k] > ans:
                ans = j - first[k]
        else:
            first[k] = j
    return ans

for a, b, exp in examples:
    print(widest_array(a, b), "(kỳ vọng", exp, ")")

3 (kỳ vọng 3 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n)` thời gian (không băm), `O(n)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1:

| j | a − b | s | s đã có? | first | ans |
|---|---|---|---|---|---|
| — | | 0 | | {0: −1} | 0 |
| 0 | 1 | 1 | không | {0: −1, 1: 0} | 0 |
| 1 | 0 | 1 | có (0) | | 1 − 0 = 1 |
| 2 | −1 | 0 | có (−1) | | 2 − (−1) = 3 |
| 3 | 1 | 1 | có (0) | | 3 − 0 = 3 |

In [12]:
def trace_hash(nums1, nums2):
    first = {0: -1}; s = ans = 0
    print(f"  khởi tạo: first = {first}")
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first:
            cand = j - first[s]; ans = max(ans, cand)
            print(f"  j={j}: d={a - b:>2}, s={s:>2} đã gặp ở {first[s]:>2} -> đoạn [{first[s] + 1}..{j}] dài {cand}, ans={ans}")
        else:
            first[s] = j
            print(f"  j={j}: d={a - b:>2}, s={s:>2} lần đầu -> first[{s}] = {j}")
    return ans

print("Ví dụ 1:"); trace_hash([1, 1, 0, 1], [0, 1, 1, 0])
print("Ví dụ 3:"); trace_hash([0], [1])

Ví dụ 1:
  khởi tạo: first = {0: -1}
  j=0: d= 1, s= 1 lần đầu -> first[1] = 0
  j=1: d= 0, s= 1 đã gặp ở  0 -> đoạn [1..1] dài 1, ans=1
  j=2: d=-1, s= 0 đã gặp ở -1 -> đoạn [0..2] dài 3, ans=3
  j=3: d= 1, s= 1 đã gặp ở  0 -> đoạn [1..3] dài 3, ans=3
Ví dụ 3:
  khởi tạo: first = {0: -1}
  j=0: d=-1, s=-1 lần đầu -> first[-1] = 0


0

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def widestPairOfIndices(self, nums1: List[int], nums2: List[int]) -> int:
        first = {0: -1}                    # hiệu tiền tố -> chỉ số j đầu tiên đạt giá trị đó
        s = ans = 0
        for j, (a, b) in enumerate(zip(nums1, nums2)):
            s += a - b
            if s in first:
                if j - first[s] > ans:
                    ans = j - first[s]
            else:
                first[s] = j
        return ans

for a, b, exp in examples:
    assert Solution().widestPairOfIndices(a, b) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Đặt `S(j) = Σ_{k ≤ j} (nums1[k] − nums2[k])`, `S(−1) = 0`. Đoạn `[i, j]` hợp lệ ⇔ `S(j) − S(i − 1) = 0` ⇔ `S(i − 1) = S(j)`.

Với `j` cố định, đoạn hợp lệ **dài nhất** kết thúc tại `j` ứng với `i − 1` **nhỏ nhất** trong `{−1, 0, …, j − 1}` có `S(i − 1) = S(j)`; đó chính là `first[S(j)]` (bất biến: `first` chứa chỉ số đầu tiên của mỗi giá trị trong `S(−1..j−1)` khi bắt đầu xử lý `j`). Ta chỉ ghi `first` khi giá trị **mới** ⇒ không bao giờ ghi đè chỉ số nhỏ hơn. Lấy `max` trên mọi `j` ⇒ đáp án. Nếu không `j` nào thấy giá trị lặp ⇒ không có đoạn hợp lệ ⇒ `0`.

Kiểm chứng bất biến bằng máy: tại mỗi bước, `first[v]` đúng bằng chỉ số **nhỏ nhất** `k ∈ {−1, …, j}` có tiền tố `S(k) = v`.

In [14]:
rnd = random.Random(11)
for _ in range(300):
    n_ = rnd.randint(1, 40)
    a = [rnd.randint(0, 1) for _ in range(n_)]; b = [rnd.randint(0, 1) for _ in range(n_)]
    S = [0] + list(accumulate(x - y for x, y in zip(a, b)))     # S[k+1] = S(k)
    first = {0: -1}; s_ = 0
    for j in range(n_):
        s_ += a[j] - b[j]
        first.setdefault(s_, j)
        for v, idx in first.items():
            assert idx == min(k for k in range(-1, j + 1) if S[k + 1] == v)
print("Bất biến 'chỉ số đầu tiên' đúng trên 300 mảng ngẫu nhiên")

Bất biến 'chỉ số đầu tiên' đúng trên 300 mảng ngẫu nhiên


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n³)` | `O(1)` |
| 2. Tiền tố + mọi cặp | `O(n²)` | `O(n)` |
| 3. Bảng băm | `O(n)` | `O(n)` |
| 4. Mảng offset | `O(n)` | `O(n)` |

In [15]:
rnd = random.Random(1983)
n = 10**5
A = [rnd.randint(0, 1) for _ in range(n)]
Bb = [rnd.randint(0, 1) for _ in range(n)]
A2 = [1] * n; B2 = [0] * (n - 1) + [1]                     # hiệu tiền tố tăng dần: gần như không lặp
for name, fn in [("hash", widest_hash), ("array", widest_array), ("Solution", Solution().widestPairOfIndices)]:
    t0 = time.perf_counter(); r1 = fn(A, Bb); r2 = fn(A2, B2); dt = time.perf_counter() - t0
    print(f"{name:<9}: ngẫu nhiên -> {r1}, khó -> {r2}  {dt * 1000:6.1f} ms")
t0 = time.perf_counter(); r = widest_prefix_pairs(A2[:3000], B2[-3000:]); dt = time.perf_counter() - t0
print(f"prefix_pairs n=3000 (trường hợp xấu): {r} ({dt * 1000:.0f} ms)")

hash     : ngẫu nhiên -> 32632, khó -> 1    22.9 ms
array    : ngẫu nhiên -> 32632, khó -> 1    14.1 ms
Solution : ngẫu nhiên -> 32632, khó -> 1    19.4 ms


prefix_pairs n=3000 (trường hợp xấu): 1 (132 ms)


## 12. Bộ test

In [16]:
ALL = {
    "prefix_pairs": widest_prefix_pairs,
    "hash": widest_hash,
    "array": widest_array,
    "Solution": lambda a, b: Solution().widestPairOfIndices(a, b),
}

def check(a, b, exp=None, brute=True):
    ref = widest_brute(a, b) if brute else widest_hash(a, b)
    for name, fn in ALL.items():
        if not brute and name == "prefix_pairs":
            continue
        assert fn(a, b) == ref, (name, a, b)
    if exp is not None:
        assert ref == exp, (a, b, ref, exp)
    return ref

for a, b, exp in examples:
    check(a, b, exp)
print("Ví dụ đề: OK")

check([1, 1, 1], [1, 1, 1], 3)            # giống hệt
check([1, 1, 1], [0, 0, 0], 0)            # không đoạn nào
check([0, 0], [0, 0], 2)
check([1, 0, 0, 1], [0, 1, 1, 0], 4)      # tự dựng: cả mảng
check([1, 0, 1, 0, 1], [0, 1, 0, 1, 0], 4)
print("Edge case: OK")

from itertools import product
cnt = 0
for n_ in range(1, 7):
    for a in product((0, 1), repeat=n_):
        for b in product((0, 1), repeat=n_):
            check(list(a), list(b)); cnt += 1
print(f"Toàn bộ {cnt} cặp mảng độ dài <= 6: OK")

rnd = random.Random(2000)
for _ in range(50):
    n_ = rnd.randint(1, 3000)
    p = rnd.random()
    check([int(rnd.random() < p) for _ in range(n_)], [int(rnd.random() < 0.5) for _ in range(n_)], brute=False)
print("50 mảng cỡ lớn: các cách khớp nhau: OK")

Ví dụ đề: OK
Edge case: OK


Toàn bộ 5460 cặp mảng độ dài <= 6: OK
50 mảng cỡ lớn: các cách khớp nhau: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên khởi tạo `first[0] = −1`

In [17]:
def widest_bug_no_base(nums1, nums2):
    first = {}                                   # SAI: thiếu tiền tố rỗng
    s = ans = 0
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first: ans = max(ans, j - first[s])
        else: first[s] = j
    return ans

a, b = [1, 0, 0, 1], [0, 1, 1, 0]
print("sai :", widest_bug_no_base(a, b), "| đúng:", Solution().widestPairOfIndices(a, b))

sai : 2 | đúng: 4


**Vì sao sai:** đoạn bắt đầu từ `i = 0` cần so với tiền tố **rỗng** (giá trị 0 ở vị trí `−1`). **Sửa:** `first = {0: -1}`.

### Lỗi 2 — Ghi đè chỉ số (lưu lần xuất hiện **cuối** thay vì đầu)

In [18]:
def widest_bug_overwrite(nums1, nums2):
    first = {0: -1}; s = ans = 0
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first:
            ans = max(ans, j - first[s])
        first[s] = j                             # SAI: ghi đè luôn
    return ans

rnd = random.Random(2)
found = None
while found is None:
    n_ = rnd.randint(2, 8)
    a = [rnd.randint(0, 1) for _ in range(n_)]; b = [rnd.randint(0, 1) for _ in range(n_)]
    if widest_bug_overwrite(a, b) != Solution().widestPairOfIndices(a, b):
        found = (a, b)
print("phản ví dụ:", found, "| ghi đè:", widest_bug_overwrite(*found), "| đúng:", Solution().widestPairOfIndices(*found))

phản ví dụ: ([0, 0, 0, 1, 0, 1, 1, 0], [0, 0, 1, 1, 1, 1, 1, 0]) | ghi đè: 1 | đúng: 3


**Vì sao sai:** muốn đoạn **dài nhất**, cần chỉ số **sớm nhất**. Ghi đè làm đoạn tìm được ngắn lại. **Sửa:** chỉ gán khi khóa chưa có.

### Lỗi 3 — Lệch một: dùng `j − first[s] + 1`

In [19]:
def widest_bug_plus_one(nums1, nums2):
    first = {0: -1}; s = ans = 0
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first: ans = max(ans, j - first[s] + 1)    # SAI: first[s] là chỉ số TRƯỚC đoạn
        else: first[s] = j
    return ans

a, b = [0, 1], [1, 1]
print("sai :", widest_bug_plus_one(a, b), "| đúng:", Solution().widestPairOfIndices(a, b))

sai : 2 | đúng: 1


**Vì sao sai:** `first[s]` là chỉ số **ngay trước** đoạn (`i − 1`), nên độ dài là `j − (i − 1) = j − first[s]`. **Sửa:** bỏ `+ 1`.

### Lỗi 4 — So sánh tổng tiền tố **riêng** của từng mảng thay vì hiệu

In [20]:
def widest_bug_separate(nums1, nums2):
    P1 = list(accumulate(nums1, initial=0)); P2 = list(accumulate(nums2, initial=0))
    first = {}; ans = 0
    for k in range(len(P1)):
        key = (P1[k], P2[k])                    # SAI: đòi hai tiền tố cùng bằng nhau từng phần
        if key in first: ans = max(ans, k - first[key])
        else: first[key] = k
    return ans

a, b = [1, 1, 0, 1], [0, 1, 1, 0]
print("sai :", widest_bug_separate(a, b), "| đúng:", Solution().widestPairOfIndices(a, b))

sai : 0 | đúng: 3


**Vì sao sai:** cặp `(P1, P2)` bằng nhau ở hai vị trí ⇔ **cả hai** tổng đoạn bằng 0 — điều kiện quá chặt. Ta chỉ cần **hiệu** `P1 − P2` bằng nhau. **Sửa:** khóa là `P1[k] − P2[k]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Hai đại lượng trên cùng một đoạn bằng nhau" ⇒ trừ đi, thành "tổng đoạn bằng 0".
- "Đoạn **dài nhất** có tổng 0 / tổng `k`" ⇒ tiền tố + bảng băm **chỉ số đầu tiên** (`{0: -1}`).
- "**Đếm** số đoạn có tổng `k`" ⇒ tiền tố + bảng băm **số lần xuất hiện** (`{0: 1}`).
- Giá trị tiền tố trong miền nhỏ ⇒ mảng với offset.
- Bài cùng khuôn: **LeetCode 525 — Contiguous Array** (số 0 bằng số 1), **325 — Maximum Size Subarray Sum Equals k**.

## 15. Biến thể của bài

1. **LeetCode 525 — Contiguous Array:** đổi `0 → −1` rồi tìm đoạn tổng 0 dài nhất.
2. **Đếm số cặp `(i, j)`** có tổng bằng nhau: `Σ C(cnt[v], 2)` trên các giá trị tiền tố.
3. **Tổng `nums1` lớn hơn `nums2` đúng `k`:** tìm `first[s − k]`.
4. **Đoạn ngắn nhất** có tổng bằng nhau (độ dài ≥ 1): lưu chỉ số **cuối cùng**.
5. **Mảng không nhị phân:** cùng thuật toán (hiệu có thể lớn ⇒ dùng `dict`).

In [21]:
def find_max_length_525(nums):
    first = {0: -1}; s = ans = 0
    for j, x in enumerate(nums):
        s += 1 if x == 1 else -1
        if s in first: ans = max(ans, j - first[s])
        else: first[s] = j
    return ans

def count_equal_pairs(nums1, nums2):
    from collections import Counter
    c = Counter(accumulate((a - b for a, b in zip(nums1, nums2)), initial=0))
    return sum(v * (v - 1) // 2 for v in c.values())

def count_equal_brute(nums1, nums2):
    n = len(nums1)
    return sum(sum(nums1[i:j + 1]) == sum(nums2[i:j + 1]) for i in range(n) for j in range(i, n))

print("525 [0,1,0] ->", find_max_length_525([0, 1, 0]))
print("đếm cặp ví dụ 1:", count_equal_pairs([1, 1, 0, 1], [0, 1, 1, 0]), "| brute:", count_equal_brute([1, 1, 0, 1], [0, 1, 1, 0]))

525 [0,1,0] -> 2
đếm cặp ví dụ 1: 4 | brute: 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi cặp, cộng lại | `O(n³)` | `O(1)` | kiểm thử |
| 2. Tiền tố + mọi cặp | `P[i] == P[j+1]` | `O(n²)` | `O(n)` | hiểu bài |
| 3. Bảng băm chỉ số đầu | `{0: -1}` | `O(n)` | `O(n)` | **nộp bài** |
| 4. Mảng offset | hiệu trong `[−n, n]` | `O(n)` | `O(n)` | nhanh hơn, không băm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().widestPairOfIndices([1, 0, 1, 1, 0], [0, 0, 1, 0, 1])`?

**Bài 2 (tính tay):** Viết `d`, `P` và bảng `first` cho `nums1 = [0, 0, 1, 1]`, `nums2 = [1, 1, 0, 0]`; tìm đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `widest_pair(nums1, nums2)` trả về **cặp chỉ số** `(i, j)` của một đoạn rộng nhất (hoặc `None`).

**Bài 4 (tự viết lại):** Viết lại cách 3 cho **mảng số nguyên bất kỳ** (không chỉ 0/1) và kiểm bằng vét cạn.

**Bài 5 (mini challenge):** Viết `widest_with_diff_k(nums1, nums2, k)` — đoạn rộng nhất mà `sum(nums1) − sum(nums2) = k` trên đoạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def widest_pair(nums1, nums2):
    # TODO: như Solution nhưng lưu (first[s] + 1, j) khi cập nhật ans
    pass

# Bài 4 — TODO
def widest_general(nums1, nums2):
    # TODO: thuật toán giống hệt (dict, vì hiệu có thể lớn)
    pass

# Bài 5 — TODO
def widest_with_diff_k(nums1, nums2, k):
    # TODO: tra first[s - k] (s - k là tiền tố trước đoạn)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `d = [1, 0, 0, 1, −1]`.
- Mức 2: `P = [0, 1, 1, 1, 2, 1]`.
- Mức 3: Giá trị 1 ở chỉ số 1 và 5 ⇒ đoạn dài 4.

**Bài 2**
- Mức 1: `d = [−1, −1, 1, 1]`.
- Mức 2: `P = [0, −1, −2, −1, 0]`.
- Mức 3: `P[0] = P[4] = 0` ⇒ cả mảng ⇒ 4.

**Bài 3**
- Mức 1: Đoạn ứng với `first[s]` là `[first[s] + 1, j]`.
- Mức 2: Lưu khi `ans` tăng.
- Mức 3: Kiểm tổng hai đoạn bằng nhau và độ dài bằng `Solution()`.

**Bài 4**
- Mức 1: Không có gì phụ thuộc vào 0/1.
- Mức 2: Dùng `dict`.
- Mức 3: Kiểm với số âm.

**Bài 5**
- Mức 1: `S(j) − S(i−1) = k` ⇔ `S(i−1) = S(j) − k`.
- Mức 2: Vẫn lưu chỉ số đầu tiên của mỗi `S`.
- Mức 3: Ghi `first[s]` sau khi tra (để `k = 0` vẫn đúng).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().widestPairOfIndices([1, 0, 1, 1, 0], [0, 0, 1, 0, 1])
print("Bài 1:", r1)
assert r1 == 4 == widest_brute([1, 0, 1, 1, 0], [0, 0, 1, 0, 1])

# Bài 2
trace_hash([0, 0, 1, 1], [1, 1, 0, 0])
assert Solution().widestPairOfIndices([0, 0, 1, 1], [1, 1, 0, 0]) == 4

# Bài 3
def widest_pair_sol(nums1, nums2):
    first = {0: -1}; s = 0; best = None
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first:
            if best is None or j - first[s] > best[1] - best[0] + 1:
                best = (first[s] + 1, j)
        else:
            first[s] = j
    return best

rnd = random.Random(3)
for _ in range(2000):
    n_ = rnd.randint(1, 12)
    a = [rnd.randint(0, 1) for _ in range(n_)]; b = [rnd.randint(0, 1) for _ in range(n_)]
    p = widest_pair_sol(a, b); w = Solution().widestPairOfIndices(a, b)
    if w == 0:
        assert p is None
    else:
        i, j = p
        assert j - i + 1 == w and sum(a[i:j + 1]) == sum(b[i:j + 1])
print("Bài 3 OK | ví dụ 1:", widest_pair_sol([1, 1, 0, 1], [0, 1, 1, 0]))

Bài 1: 4
  khởi tạo: first = {0: -1}
  j=0: d=-1, s=-1 lần đầu -> first[-1] = 0
  j=1: d=-1, s=-2 lần đầu -> first[-2] = 1
  j=2: d= 1, s=-1 đã gặp ở  0 -> đoạn [1..2] dài 2, ans=2
  j=3: d= 1, s= 0 đã gặp ở -1 -> đoạn [0..3] dài 4, ans=4
Bài 3 OK | ví dụ 1: (0, 2)


In [24]:
# Bài 4
def widest_general_sol(nums1, nums2):
    first = {0: -1}; s = ans = 0
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s in first: ans = max(ans, j - first[s])
        else: first[s] = j
    return ans

# Bài 5
def widest_with_diff_k_sol(nums1, nums2, k):
    first = {0: -1}; s = ans = 0
    for j, (a, b) in enumerate(zip(nums1, nums2)):
        s += a - b
        if s - k in first:
            ans = max(ans, j - first[s - k])
        first.setdefault(s, j)
    return ans

def diff_k_brute(nums1, nums2, k):
    n = len(nums1)
    return max([j - i + 1 for i in range(n) for j in range(i, n) if sum(nums1[i:j + 1]) - sum(nums2[i:j + 1]) == k] or [0])

rnd = random.Random(5)
for _ in range(1500):
    n_ = rnd.randint(1, 10)
    a = [rnd.randint(-3, 3) for _ in range(n_)]; b = [rnd.randint(-3, 3) for _ in range(n_)]
    assert widest_general_sol(a, b) == widest_brute(a, b)
    k = rnd.randint(-4, 4)
    assert widest_with_diff_k_sol(a, b, k) == diff_k_brute(a, b, k)
print("Bài 4, Bài 5 OK | diff k=1 ví dụ 1:", widest_with_diff_k_sol([1, 1, 0, 1], [0, 1, 1, 0], 1))

Bài 4, Bài 5 OK | diff k=1 ví dụ 1: 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đoạn dài nhất có tổng (hiệu) bằng 0":**

```python
first = {0: -1}
s = ans = 0
for j, x in enumerate(values):        # values = nums1[j] - nums2[j]
    s += x
    if s in first:
        ans = max(ans, j - first[s])  # đoạn [first[s] + 1, j]
    else:
        first[s] = j                  # chỉ ghi lần ĐẦU
return ans
```

**Checklist:**
- [ ] Làm trên **hiệu** `nums1 − nums2`?
- [ ] `first = {0: -1}` (tiền tố rỗng)?
- [ ] Chỉ ghi chỉ số **lần đầu** (không ghi đè)?
- [ ] Độ dài `j − first[s]` (không `+1`)?
- [ ] Không có đoạn ⇒ trả `0`?
- [ ] Chữ ký đúng: `def widestPairOfIndices(self, nums1: List[int], nums2: List[int]) -> int`?